# Scaling laws: Random / K-Centers / Herding

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import json
import sys
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import seaborn as sns
import torch
from transformers import set_seed

cwd = Path.cwd().resolve()
if (cwd / "notebooks" / "dilm_wrapper.py").exists():
    ROOT = cwd
    NOTEBOOK_DIR = cwd / "notebooks"
elif (cwd / "dilm_wrapper.py").exists():
    NOTEBOOK_DIR = cwd
    ROOT = cwd.parent
else:
    raise RuntimeError(f"Cannot locate project root from {cwd}")

sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(ROOT / "src"))

from dataset_attrs import DATASET_ATTRS
from learner import MODEL_ATTRS
from dilm_wrapper import (
    RESULTS_ROOT,
    build_coreset_module,
    build_data_module,
    build_evaluator,
    build_generator,
    build_learner,
    save_summary,
    summarize,
)

sns.set_theme(style="whitegrid")

In [ ]:
METHODS = ["random", "k_centers", "herding"]
SEED = 42

TASKS = ["sst2", "ag_news"]
LEARNER_MODELS = [
    "bert-base-uncased",
    "roberta-base",
    "albert-base-v2",
]

# 3 methods × 2 tasks × 3 models × 6 dpc = 108 runs
# ~40 sec/run на H100 = ~72 мин обучение
# embeddings кешируются на диск → считаются 1 раз на (model, task, class)
DPC_GRID = [1, 5, 10, 50, 100, 500]
N_DATASET = 1
SKIP_EXISTING = True

EVAL_KW = dict(
    train_step=200,
    batch_size=128,   # H100 держит без проблем
    lr=1e-4,
    n_eval_per_dataset=1,
    bf16=True,
)

SCALING_ROOT = RESULTS_ROOT / "scaling_laws"
SCALING_ROOT.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
mlflow.set_tracking_uri(f"file:{RESULTS_ROOT}/mlruns")
mlflow.set_experiment("scaling_laws.all_methods")

print("methods:", METHODS)
print("tasks:", TASKS)
print("models:", LEARNER_MODELS)
print("dpc:", DPC_GRID)
print("total runs:", len(METHODS) * len(TASKS) * len(LEARNER_MODELS) * len(DPC_GRID))

In [ ]:
def safe_name(value: str) -> str:
    return value.replace("/", "__")


def run_dir(method: str, task: str, learner_model: str, dpc: int) -> Path:
    return SCALING_ROOT / method / task / safe_name(learner_model) / f"dpc{dpc}_seed{SEED}"


def summary_path(method: str, task: str, learner_model: str, dpc: int) -> Path:
    return run_dir(method, task, learner_model, dpc) / "summary.json"


def load_summary(path: Path) -> dict:
    return json.loads(path.read_text())


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def run_method_task_model(method: str, task: str, learner_model: str) -> list[dict]:
    print(f"\n=== {method} | {task} | {learner_model} ===")
    rows = []

    learner = build_learner(learner_model, task, gradient_checkpointing=True)
    generator = build_generator(task, gradient_checkpointing=True)
    data_module = build_data_module(task, learner, generator, train_batch_size=64)
    evaluator = build_evaluator(task, **EVAL_KW)
    selector = build_coreset_module(task, method, data_module)
    metric_key = evaluator.metric_key

    for dpc in DPC_GRID:
        path = summary_path(method, task, learner_model, dpc)
        if SKIP_EXISTING and path.exists():
            print(f"  skip existing: dpc={dpc}")
            rows.append(load_summary(path))
            continue

        k = dpc * data_module.num_labels
        rd = run_dir(method, task, learner_model, dpc)
        rd.mkdir(parents=True, exist_ok=True)
        print(f"  run dpc={dpc}, k={k}")

        t0 = time.perf_counter()
        distilled = selector.generate_dataset(dpc=dpc, n=N_DATASET)
        selection_time_sec = time.perf_counter() - t0

        t1 = time.perf_counter()
        with mlflow.start_run(run_name=f"{method}.{task}.{safe_name(learner_model)}.dpc{dpc}.seed{SEED}"):
            mlflow.log_params({
                "method": method,
                "task": task,
                "learner": learner_model,
                "dpc": dpc,
                "k": k,
                "n_dataset": N_DATASET,
                "seed": SEED,
                **EVAL_KW,
            })
            results = evaluator.evaluate(
                dataset_list=distilled,
                learner=learner,
                data_module=data_module,
                save_result_dir=str(rd / "metrics"),
                verbose=True,
            )
        eval_time_sec = time.perf_counter() - t1

        summary = summarize(
            results,
            metric_key,
            name=f"{method}_dpc{dpc}_seed{SEED}",
            method=method,
            task=task,
            learner=learner_model,
            dpc=dpc,
            k=k,
            n_dataset=N_DATASET,
            seed=SEED,
            selection_time_sec=selection_time_sec,
            eval_time_sec=eval_time_sec,
        )
        save_summary(summary, rd)
        rows.append(summary)
        print(f"  {metric_key}={summary[f'{metric_key}_mean']:.4f}  eval_time={eval_time_sec:.0f}s")

    del learner, generator, data_module, evaluator, selector
    cleanup_cuda()
    return rows

## Run

In [ ]:
all_rows = []
errors = []

for method in METHODS:
    for task in TASKS:
        for learner_model in LEARNER_MODELS:
            try:
                all_rows.extend(run_method_task_model(method, task, learner_model))
            except Exception as exc:
                cleanup_cuda()
                err = {
                    "method": method,
                    "task": task,
                    "learner": learner_model,
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "traceback": traceback.format_exc(),
                }
                errors.append(err)
                err_dir = SCALING_ROOT / method / task / safe_name(learner_model)
                err_dir.mkdir(parents=True, exist_ok=True)
                (err_dir / "error.json").write_text(json.dumps(err, indent=2))
                print("FAILED", method, task, learner_model, type(exc).__name__, exc)

df = pd.DataFrame(all_rows)
csv_path = SCALING_ROOT / "scaling_laws_all_methods.csv"
df.to_csv(csv_path, index=False)
print("saved", csv_path)
print("errors:", len(errors))
df

## Collect saved summaries

In [ ]:
rows = [json.loads(path.read_text()) for path in SCALING_ROOT.rglob("summary.json")]
df = pd.DataFrame(rows)
if not df.empty:
    df["score"] = df.apply(lambda row: row[f"{row['metric']}_mean"], axis=1)
    df = df.sort_values(["method", "task", "learner", "dpc"])
    df.to_csv(SCALING_ROOT / "scaling_laws_all_methods.csv", index=False)
df

## Plot

In [ ]:
if df.empty:
    raise RuntimeError("No summaries found. Run previous cells first.")

plot_df = df.copy()
plot_df["learner_short"] = plot_df["learner"].str.split("/").str[-1]

# Plot 1: методы сравниваем на bert-base, col=task
bert_df = plot_df[plot_df["learner"] == "bert-base-uncased"]
g1 = sns.relplot(
    data=bert_df,
    x="k",
    y="score",
    hue="method",
    col="task",
    kind="line",
    marker="o",
    facet_kws={"sharey": False, "sharex": False},
    height=4,
    aspect=1.2,
)
g1.set_axis_labels("K (примеров всего)", "Accuracy")
g1.set_titles("{col_name}")
g1.fig.suptitle("Scaling laws: сравнение методов (bert-base-uncased)", y=1.03)
plot1_path = SCALING_ROOT / "scaling_laws_methods_bert.png"
g1.fig.savefig(plot1_path, dpi=160, bbox_inches="tight")
print("saved", plot1_path)
plt.show()

# Plot 2: архитектуры сравниваем на herding, col=task
herding_df = plot_df[plot_df["method"] == "herding"]
g2 = sns.relplot(
    data=herding_df,
    x="k",
    y="score",
    hue="learner_short",
    col="task",
    kind="line",
    marker="o",
    facet_kws={"sharey": False, "sharex": False},
    height=4,
    aspect=1.2,
)
g2.set_axis_labels("K (примеров всего)", "Accuracy")
g2.set_titles("{col_name}")
g2.fig.suptitle("Scaling laws: сравнение архитектур (herding)", y=1.03)
plot2_path = SCALING_ROOT / "scaling_laws_models_herding.png"
g2.fig.savefig(plot2_path, dpi=160, bbox_inches="tight")
print("saved", plot2_path)
plt.show()